# Crib sheet: data access, pandas and plotting

A quick reference for the tools used in the Fanal exercise.
Covers HDF5 data reading, pandas DataFrames, boolean masks (selections), and histogram plotting with `pltext`.

## Setup

In [ ]:
%matplotlib inline

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)

import core.pltext as pltext
import core.utils  as ut
pltext.style()

print('Fanal root:', rootpath)

---
## 1. Reading HDF5 data

The data is stored in HDF5 files (`.h5`).  Each file contains several *keys* (tables):

| Key | Content |
|-----|----------|
| `mc/bi214` | Simulated $^{214}$Bi background |
| `mc/tl208` | Simulated $^{208}$Tl background |
| `mc/bb0nu` | Simulated $\beta\beta0\nu$ signal |
| `data/blind` | Real data with signal region blinded |
| `data/roi` | Real data in the Region of Interest |

Use `pd.read_hdf()` to load a table into a pandas DataFrame.  
Always append `.fillna(0.)` to replace missing values (NaN) with zero.

In [ ]:
# Read one MC sample
dirpath  = rootpath + '/data/'
filename = 'fanal_new_gamma.h5'

mcbi = pd.read_hdf(dirpath + filename, key = 'mc/bi214').fillna(0.)
print('Loaded {:d} events'.format(len(mcbi)))

---
## 2. pandas DataFrame basics

A DataFrame is a table: each row is an event, each column is a variable.

In [ ]:
# Show the first 5 rows
mcbi.head()

In [ ]:
# List all column names
print(mcbi.columns.tolist())

In [ ]:
# Access a single column (returns a pandas Series)
energies = mcbi.E
print(type(energies))
print('First 3 values:', energies.values[:3])

In [ ]:
# Number of events
print('Total events:', len(mcbi))

In [ ]:
# Quick summary statistics
mcbi[['E', 'num_tracks', 'blob2_E']].describe()

---
## 3. Boolean masks and selections

A **boolean mask** is an array of `True`/`False` values, one per event.  
Use it to select events that satisfy a condition.

In [ ]:
# Create a mask: events with energy in [2.4, 2.7) MeV
mask_E = (mcbi.E >= 2.4) & (mcbi.E < 2.7)
print('Type :', type(mask_E))
print('Shape:', mask_E.shape)
print('First 5 values:', mask_E.values[:5])

In [ ]:
# Count how many events pass the cut
print('Events in energy range:', mask_E.sum())
print('Fraction              : {:.4f}'.format(mask_E.mean()))

In [ ]:
# Apply the mask to filter the DataFrame
mcbi_sel = mcbi[mask_E]
print('Selected events:', len(mcbi_sel))

# Or filter a single column
energies_sel = mcbi.E[mask_E]

### Combining masks

Use `&` (AND), `|` (OR), `~` (NOT).  Always wrap each condition in parentheses.

In [ ]:
# Sequential cuts
mask_ntracks = (mcbi.num_tracks == 1)
mask_blob2   = (mcbi.blob2_E > 0.4)

# Combine: energy range AND 1 track AND blob2 cut
mask_all = mask_E & mask_ntracks & mask_blob2
print('Events passing all cuts:', mask_all.sum())

### Computing an efficiency

In [ ]:
# Efficiency = fraction of events passing the cut
eff = mask_all.sum() / len(mcbi)
print('Selection efficiency: {:.4f} ({:.2f} %)'.format(eff, 100 * eff))

### Building cuts progressively

A common pattern: apply cuts one by one and track the efficiency at each step.

In [ ]:
cuts = [
    ('Energy range',  (mcbi.E >= 2.4) & (mcbi.E < 2.7)),
    ('1 track',       mcbi.num_tracks == 1),
    ('Blob2 > 0.4',   mcbi.blob2_E > 0.4),
    ('RoI',          (mcbi.E >= 2.43) & (mcbi.E < 2.48)),
]

mask = pd.Series(True, index=mcbi.index)
print('{:20s} {:>8s} {:>8s}'.format('Cut', 'Events', 'Eff (%)'))
print('-' * 40)
for name, cut in cuts:
    mask = mask & cut
    print('{:20s} {:8d} {:8.4f}'.format(name, mask.sum(), 100 * mask.mean()))

### Utility shortcut: `ut.in_range()`

In [ ]:
# Equivalent to (mcbi.E >= 2.4) & (mcbi.E < 2.7)
mask_E2 = ut.in_range(mcbi.E, (2.4, 2.7))
print('Same result:', (mask_E == mask_E2).all())

---
## 4. Plotting histograms with `pltext`

`pltext.hist()` is a wrapper around `plt.hist()` that adds a stats box (N, mean, std) to the legend and defaults to step-style histograms.

### Basic histogram

In [ ]:
pltext.hist(mcbi.E, 100, range = (2.2, 2.8))
plt.xlabel('Energy (MeV)')
plt.title('Bi-214 energy spectrum');

### Overlaying histograms from different samples

In [ ]:
# Load all samples
mctl = pd.read_hdf(dirpath + filename, key = 'mc/tl208').fillna(0.)
mcbb = pd.read_hdf(dirpath + filename, key = 'mc/bb0nu').fillna(0.)

samples = [mcbi, mctl, mcbb]
labels  = [r'$^{214}$Bi', r'$^{208}$Tl', r'$\beta\beta0\nu$']

for df, label in zip(samples, labels):
    sel = (df.E >= 2.4) & (df.E < 2.7) & (df.num_tracks == 1)
    pltext.hist(df.E[sel], 60, range = (2.4, 2.7),
               label = label, density = True, lw = 2)

plt.xlabel('Energy (MeV)')
plt.legend();

### Multi-panel figures with `pltext.canvas()`

`pltext.canvas(n)` creates a figure with `n` subplots and returns a function `subplot(i)` to activate each one (1-indexed).

In [ ]:
subplot = pltext.canvas(4)   # 4 panels, 2 columns by default

subplot(1)
pltext.hist(mcbi.E, 100, range = (2.2, 2.8))
plt.xlabel('Energy (MeV)'); plt.title('Bi: energy')

subplot(2)
pltext.hist(mcbi.num_tracks, 10, range = (0, 10))
plt.xlabel('Number of tracks'); plt.title('Bi: tracks')

subplot(3)
sel = (mcbi.num_tracks == 1)
pltext.hist(mcbi.blob2_E[sel], 100)
plt.xlabel('Blob2 energy (MeV)'); plt.title('Bi: blob2 (1 track)')

subplot(4)
pltext.hist(mcbi.track0_length[sel], 100)
plt.xlabel('Track length (mm)'); plt.title('Bi: length (1 track)')

plt.tight_layout();

### Key `pltext.hist()` parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `bins` | -- | Number of bins (required) |
| `range` | auto | Tuple `(xmin, xmax)` |
| `density` | `False` | Normalize to probability density |
| `label` | `''` | Legend label |
| `lw` | `1` | Line width |
| `stats` | `True` | Append N/mean/std to legend |
| `ylog` | `False` | Logarithmic y-axis |

### Using plain `plt.hist()`

Sometimes you need the raw `matplotlib` histogram (e.g. to get bin contents).

In [ ]:
# plt.hist returns (counts, bin_edges, patches)
counts, edges, _ = plt.hist(mcbi.E, bins=50, range=(2.2, 2.8),
                            histtype='step', lw=2)
plt.xlabel('Energy (MeV)')
plt.ylabel('Counts')

# Access bin centres and counts
centres = 0.5 * (edges[:-1] + edges[1:])
print('Bin with most counts: {:.3f} MeV ({:.0f} events)'.format(
    centres[np.argmax(counts)], counts.max()));

---
## Summary of common patterns

```python
# Read data
df = pd.read_hdf(path, key = 'mc/bi214').fillna(0.)

# Inspect
df.head()              # first rows
len(df)                # number of events
df.columns.tolist()    # column names

# Select
mask = (df.E >= 2.4) & (df.E < 2.7) & (df.num_tracks == 1)
df_sel = df[mask]      # filtered DataFrame
vals   = df.E[mask]    # filtered column

# Efficiency
eff = mask.sum() / len(df)

# Plot
pltext.hist(df.E[mask], bins, range=(2.4, 2.7),
            label='Bi', density=True, lw=2)
plt.xlabel('Energy (MeV)')
plt.legend()
```